## >>> VERSION: 2026-07-25  ·  dit-then-rcdnet-v1  <<<
**DiT -> RCDNet cascade on the full 500-image benchmark (INSTRUMENTED).**

This runs RCDNet **on the saved DiT outputs** (it does NOT re-run DiT). The attached
dataset `dit_rcdnet_input` has `Drop/` = the 500 DiT-restored images and `Clear/` = the
original ground truth. RCDNet cleans `Drop/` (the DiT output) and the result is scored
against `Clear/` (the true clean image) -> that is the DiT->RCDNet cascade result.

Same fast ~30 min single pass as RCDNet-alone (no batching). The eval line is IDENTICAL
to the working 150-study cascade notebook; only metrics/stats cells are added.

All metrics at a fixed **256x256** (same scorer/resolution as DiT-alone and RCDNet-alone).

**Attach as inputs:** RaindropClarity code, the rcdnet code (with `spa_model_best.pt`
+ `init_kernel.mat`), the **`dit_rcdnet_input`** dataset (Drop/ = DiT outputs, Clear/ = GT),
and `score_pairs.py`. Do NOT also attach the raw 500 dataset (only ONE Drop/+Clear/ folder
should be attached, or it may pick the wrong one). **GPU T4 x2, Internet on.**

**Runtime note:** the time captured here is the **RCDNet stage only**. The full cascade
cost = DiT-alone time + this RCDNet-stage time (add them when reporting).

**After it finishes:** download `metrics.zip` and `dit_rcdnet_outputs.zip`.

In [ ]:
# --- setup: copy RaindropClarity (for utils + score_pairs) and the rcdnet code ---
import os, shutil, glob
src = None
for d, _, files in os.walk('/kaggle/input'):
    if 'eval_diffusion_day_dit.py' in files:
        src = d; break
assert src, 'Could not find the RaindropClarity code under /kaggle/input'
shutil.copytree(src, '/kaggle/working/RaindropClarity', dirs_exist_ok=True)

rcd = None
for d, _, files in os.walk('/kaggle/input'):
    if 'rcdnet_cascade.py' in files:
        rcd = d; break
assert rcd, 'Could not find rcdnet_cascade.py under /kaggle/input'
shutil.copytree(rcd, '/kaggle/working/rcdnet', dirs_exist_ok=True)

os.chdir('/kaggle/working/RaindropClarity')
print('cwd    :', os.getcwd())
print('rcdnet :', rcd)
for s in glob.glob('/kaggle/input/**/score_pairs.py', recursive=True):
    shutil.copy(s, 'score_pairs.py'); print('got', s)
os.makedirs('/kaggle/working/metrics', exist_ok=True)

In [ ]:
!pip install lpips -q

In [ ]:
# --- locate the cascade input (Drop/ = DiT outputs, Clear/ = GT) and the SPA checkpoint ---
import os, glob, torch
DATA = None
for d, subs, _ in os.walk('/kaggle/input'):
    if 'Drop' in subs and 'Clear' in subs:
        DATA = d; break
assert DATA, "Couldn't find Drop/ and Clear/ under /kaggle/input"
spa = glob.glob('/kaggle/input/**/spa_model_best.pt', recursive=True)
if not spa:
    spa = glob.glob('/kaggle/working/**/spa_model_best.pt', recursive=True)
assert spa, "Couldn't find spa_model_best.pt (attach the rcdnet code dataset)"
os.environ['DATA'] = DATA
os.environ['SPA']  = spa[0]
n = len(glob.glob(os.path.join(DATA, 'Drop', '**', '*.png'), recursive=True))
print('DATA   :', DATA, '(Drop/ = DiT outputs, Clear/ = ground truth)')
print('SPA    :', spa[0])
print('images :', n, '(expected 500)')
print('GPU    :', torch.cuda.is_available(), ' device_count:', torch.cuda.device_count())

## Plumbing smoke test (20 images) — confirms score_pairs + LPIPS + CSV work

In [ ]:
# 20-image smoke test on Drop vs Clear: verifies scoring plumbing in ~1 min before the run
!python score_pairs.py --pred "$DATA/Drop" --gt "$DATA/Clear" \
    --name _smoketest --out_dir /kaggle/working/metrics --limit 20
import pandas as pd
df = pd.read_csv('/kaggle/working/metrics/_smoketest_per_image.csv')
print('\nsmoke test rows:', len(df), '(expect 20)')
assert len(df) == 20, 'smoke test did not produce 20 rows - fix before continuing'
print('PLUMBING OK')

## Static resource stats — RCDNet parameter count + model size (no GPU)

In [ ]:
# param count + checkpoint size of the RCDNet (stage-2) model, read robustly from
# spa_model_best.pt (handles state_dict / wrapped dict / saved nn.Module - never crashes)
import torch, os, csv, glob
spa = os.environ['SPA']
size_mb = os.path.getsize(spa) / (1024 * 1024)
ck = torch.load(spa, map_location='cpu', weights_only=False)
state = ck
if hasattr(state, 'state_dict'):
    state = state.state_dict()
if isinstance(state, dict):
    for k in ('state_dict', 'model', 'net', 'params', 'G', 'generator'):
        if k in state and isinstance(state[k], dict):
            state = state[k]; break
n_params = sum(v.numel() for v in state.values() if torch.is_tensor(v)) if isinstance(state, dict) else -1
os.makedirs('/kaggle/working/metrics', exist_ok=True)
with open('/kaggle/working/metrics/resource_static.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['model', 'param_count', 'param_count_millions', 'model_size_MB'])
    w.writerow(['DiT_then_RCDNet_stage2', n_params, round(n_params / 1e6, 3), round(size_mb, 2)])
print(f'RCDNet stage-2: params={n_params:,} ({n_params/1e6:.2f}M)  size={size_mb:.2f} MB')
print(open('/kaggle/working/metrics/resource_static.csv').read())

## DiT -> RCDNet cascade (the ~30 min RCDNet stage) — timed, with peak-GPU logging

In [ ]:
# RCDNet run on the DiT outputs via the EXACT working ! invocation (only out_dir renamed).
# Python timing + an nvidia-smi poller wrap around it to capture time + peak GPU memory.
import subprocess, time, os, glob
mem_log = '/kaggle/working/metrics/gpu_mem_dit_rcdnet.csv'
logf = open(mem_log, 'w')
poller = subprocess.Popen(
    ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits', '-l', '1'],
    stdout=logf)
t0 = time.time()
!cd /kaggle/working/rcdnet && python rcdnet_cascade.py --in_dir "$DATA" --out_dir /kaggle/working/dit_then_rcd --model spa_model_best.pt
elapsed = time.time() - t0
poller.terminate(); logf.close()
n_out = len(glob.glob('/kaggle/working/dit_then_rcd/Drop/**/*.png', recursive=True))
print('DiT->RCDNet stage done. elapsed:', round(elapsed, 1), 's | images restored:', n_out)
os.environ['RCD_ELAPSED'] = str(elapsed)
os.environ['RCD_N'] = str(max(n_out, 1))

In [ ]:
# runtime resource stats -> resource_runtime.csv  (RCDNet stage only; add DiT-alone time
# for the full cascade cost). real full-500 total, not projected.
import csv as _csv, os
elapsed = float(os.environ['RCD_ELAPSED'])
n = int(os.environ['RCD_N'])
mem_vals = [int(l.strip()) for l in open('/kaggle/working/metrics/gpu_mem_dit_rcdnet.csv') if l.strip().isdigit()]
peak = max(mem_vals) if mem_vals else -1
per_image = elapsed / n
row = {'model': 'DiT_then_RCDNet_stage2',
       'per_image_s': round(per_image, 4),
       'throughput_img_per_s': round(1.0 / per_image, 4),
       'total_time_s': round(elapsed, 2),
       'peak_gpu_mem_MB': peak,
       'n_images': n}
rt = '/kaggle/working/metrics/resource_runtime.csv'
with open(rt, 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=list(row.keys()))
    w.writeheader(); w.writerow(row)
print(row)
print(open(rt).read())

In [ ]:
# score cascade outputs at 256x256 -> dit_then_rcdnet_per_image.csv + summary
# (pred = RCDNet-cleaned DiT output, gt = original ground truth)
import os
os.system('python score_pairs.py --pred /kaggle/working/dit_then_rcd/Drop '
          '--gt /kaggle/working/dit_then_rcd/Clear --name dit_then_rcdnet '
          '--out_dir /kaggle/working/metrics')
print(open('/kaggle/working/metrics/dit_then_rcdnet_summary.txt').read())

In [ ]:
# collect the 500 cascade outputs into dit_rcdnet_outputs/ and zip
import shutil, os, glob
shutil.rmtree('/kaggle/working/dit_rcdnet_outputs', ignore_errors=True)
shutil.copytree('/kaggle/working/dit_then_rcd/Drop', '/kaggle/working/dit_rcdnet_outputs')
os.system('cd /kaggle/working && rm -f dit_rcdnet_outputs.zip && zip -r dit_rcdnet_outputs.zip dit_rcdnet_outputs -q')
n = len(glob.glob('/kaggle/working/dit_rcdnet_outputs/**/*.png', recursive=True))
print(f'wrote dit_rcdnet_outputs.zip ({n} images)')

## Package + verify

In [ ]:
import os
os.system('cd /kaggle/working && rm -f metrics.zip && zip -r metrics.zip metrics -q')
print('wrote metrics.zip')
print('DOWNLOAD from Output panel: metrics.zip (CSVs + stats), dit_rcdnet_outputs.zip (cascade images)')

In [ ]:
# final verification - report what exists and row counts (never crashes)
import os
try:
    import pandas as pd
except Exception:
    pd = None
m = '/kaggle/working/metrics'
def chk(p, expect=None):
    ok = os.path.exists(p); extra = ''
    if ok and p.endswith('.csv') and pd is not None:
        try:
            n = len(pd.read_csv(p)); extra = ' (%d rows)' % n
            if expect and n != expect: extra += '  << expected %d' % expect
        except Exception as e:
            extra = ' (unreadable: %s)' % e
    print(('OK   ' if ok else 'MISS ') + p + extra)
chk(m + '/dit_then_rcdnet_per_image.csv', 500)
chk(m + '/dit_then_rcdnet_summary.txt')
chk(m + '/resource_static.csv')
chk(m + '/resource_runtime.csv')
print('\ndit_then_rcdnet_per_image.csv should have 500 rows. All metrics at 256x256.')
print('Runtime here is the RCDNet stage only - add DiT-alone time for full cascade cost.')